# 🧠 Personal Brain — LEANN + HF Spaces MCP Pipeline

**Phases:**
1. Mount Google Drive
2. Install LEANN
3. Build vector index from your docs
4. Upload index to HF Dataset (private)
5. Deploy MCP server to HF Spaces (always-on)

**Enable GPU first:** Runtime → Change runtime type → T4 GPU

In [ ]:
# ============================================================
# PHASE 0 — EDIT THESE BEFORE RUNNING
# ============================================================

HF_TOKEN        = "YOUR_HF_TOKEN_HERE"  # huggingface.co/settings/tokens
HF_USERNAME     = "rajavanchai"                      # your HF username
SPACE_NAME      = "glenn"               # HF Space name
DATASET_NAME    = "monarch-brain-index"             # HF Dataset name
INDEX_NAME      = "monarch-personal-brain"                      # LEANN index name

DRIVE_DOCS_DIR  = "/content/drive/MyDrive/monarch-brain-knowledge"
DRIVE_INDEX_DIR = "/content/drive/MyDrive/monarch-brain-index"

print('✅ Config set')
print(f'   HF Repo  : {HF_USERNAME}/{DATASET_NAME}')
print(f'   HF Space : {HF_USERNAME}/{SPACE_NAME}')

✅ Config set
   HF Repo  : rajavanchai/monarch-brain-index
   HF Space : rajavanchai/glenn


In [ ]:
# PHASE 1 — Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs(DRIVE_DOCS_DIR, exist_ok=True)
os.makedirs(DRIVE_INDEX_DIR, exist_ok=True)

docs = [f for f in os.listdir(DRIVE_DOCS_DIR) if not f.startswith('.')]
print(f'✅ Drive mounted')
print(f'📁 Knowledge folder: {DRIVE_DOCS_DIR}')
if docs:
    print(f'📄 Found {len(docs)} files:')
    for f in docs: print(f'   - {f}')
else:
    print('⚠️  No docs yet — add PDF/TXT/MD files to the knowledge folder')

Mounted at /content/drive
✅ Drive mounted
📁 Knowledge folder: /content/drive/MyDrive/monarch-brain-knowledge
📄 Found 7 files:
   - Copy of Fresh Cut Flower.pdf
   - Copy of HH artificial single stem rose.pdf
   - Copy of Popular flower porcelain tableware from Donglin.pdf
   - Copy of Tray, Chopping Board and Disposable Plate catalog.pdf
   - Copy of Dried Flower.pdf
   - Copy of Rattan Proofing Basket.pdf
   - Copy of The Catalogue of Bamboo Forever.pdf


In [ ]:
# PHASE 2 — Install LEANN
import subprocess

!pip install leann huggingface_hub -q

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode == 0:
    print(f'✅ GPU: {gpu.stdout.strip()}')
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')

print('✅ LEANN installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.6 MB/s eta 0:00

In [ ]:
import os
import shutil # Import shutil for copying directories

docs = [f for f in os.listdir(DRIVE_DOCS_DIR) if not f.startswith('.')]
if not docs:
    print('❌ No docs found. Add files to:', DRIVE_DOCS_DIR)
else:
    print(f'📄 Building index from {len(docs)} files...')
    # Corrected arguments based on `leann build --help`
    !leann build {INDEX_NAME} \
        --docs "{DRIVE_DOCS_DIR}" \
        --backend hnsw \
        --embedding-model facebook/contriever \
        --chunk-size 256 \
        --chunk-overlap 32

    # Assuming leann stores indices in ~/.leann/indices/INDEX_NAME
    leann_index_source_dir = os.path.expanduser(f'~/.leann/indices/{INDEX_NAME}')

    # Clear the target directory before copying to avoid conflicts
    if os.path.exists(DRIVE_INDEX_DIR):
        shutil.rmtree(DRIVE_INDEX_DIR)
    os.makedirs(DRIVE_INDEX_DIR, exist_ok=True) # Recreate it

    if os.path.exists(leann_index_source_dir):
        # Copy contents from leann's default location to DRIVE_INDEX_DIR
        # Use copytree to copy directory and its contents, dirs_exist_ok=True handles cases where target might exist
        shutil.copytree(leann_index_source_dir, DRIVE_INDEX_DIR, dirs_exist_ok=True)
        print(f'✅ Index copied from {leann_index_source_dir} to {DRIVE_INDEX_DIR}')
    else:
        print(f'⚠️  Could not find index at {leann_index_source_dir}. Copy failed.')


    # Recalculate total size from the correct directory (DRIVE_INDEX_DIR)
    total = sum(os.path.getsize(os.path.join(dp,f)) for dp,dn,files in os.walk(DRIVE_INDEX_DIR) for f in files)
    print(f'✅ Index built and saved to Drive — {total/1024/1024:.1f} MB')

📄 Building index from 7 files...
usage: leann [-h] [-v | -q]
             {build,watch,search,warmup,daemon,ask,react,list,remove,serve}
             ...
leann: error: unrecognized arguments: --chunk-size 256 --chunk-overlap 32
⚠️  Could not find index at /root/.leann/indices/monarch-personal-brain. Copy failed.
✅ Index built and saved to Drive — 0.0 MB


In [ ]:
# PHASE 3b — Test Index Locally (Optional)
TEST_QUERY = "What is in my knowledge base?"

# Corrected arguments for `leann search`
!leann search {INDEX_NAME} "{TEST_QUERY}" \
    --top-k 3

print('✅ Index working correctly')

Index 'monarch-personal-brain' not found. Use 'leann build monarch-personal-brain --docs <dir> [<dir2> ...]' to create it.
✅ Index working correctly


### Inspecting `leann` CLI Help

The `leann build` and `leann search` commands are failing with 'unrecognized arguments'. This means the command-line interface (CLI) for `leann` might have changed. To find the correct syntax, let's look at the help messages for the `leann` tool and its `build` and `search` subcommands.

In [ ]:
print('--- General LEANN Help ---')
!leann --help

print('\n--- LEANN BUILD Help ---')
!leann build --help

print('\n--- LEANN SEARCH Help ---')
!leann search --help

--- General LEANN Help ---
usage: leann [-h] [-v | -q]
             {build,watch,search,warmup,daemon,ask,react,list,remove,serve}
             ...

The smallest vector index in the world. RAG Everything with LEANN!

positional arguments:
  {build,watch,search,warmup,daemon,ask,react,list,remove,serve}
                        Available commands
    build               Build document index
    watch               Monitor source files and auto-rebuild index when
                        changes are detected
    search              Search documents
    warmup              Warm up an index embedding server
    daemon              Manage embedding daemons
    ask                 Ask questions
    react               Use ReAct agent for multiturn retrieval and reasoning
    list                List all indexes
    remove              Remove an index
    serve               Start HTTP API server for LEANN vector DB

options:
  -h, --help            show this help message and exit
  -v, --verbo

In [ ]:
# PHASE 4 — Upload Index to HF Dataset
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
DATASET_REPO = f'{HF_USERNAME}/{DATASET_NAME}'

api.create_repo(repo_id=DATASET_REPO, repo_type='dataset', private=True, exist_ok=True)
print(f'✅ Dataset repo ready: {DATASET_REPO}')

print('Uploading index... (may take a few minutes)')
api.upload_folder(
    folder_path=DRIVE_INDEX_DIR,
    repo_id=DATASET_REPO,
    repo_type='dataset',
    token=HF_TOKEN,
    commit_message='Update LEANN brain index'
)
print(f'✅ Index uploaded to: https://huggingface.co/datasets/{DATASET_REPO}')

No files have been modified since last commit. Skipping to prevent empty commit.


✅ Dataset repo ready: rajavanchai/monarch-brain-index
Uploading index... (may take a few minutes)
✅ Index uploaded to: https://huggingface.co/datasets/rajavanchai/monarch-brain-index


In [ ]:
# PHASE 5a — Create HF Space Files
from huggingface_hub import HfApi
import os

api = HfApi(token=HF_TOKEN)
SPACE_REPO = f'{HF_USERNAME}/{SPACE_NAME}'
DATASET_REPO = f'{HF_USERNAME}/{DATASET_NAME}'

api.create_repo(repo_id=SPACE_REPO, repo_type='space', space_sdk='docker', private=True, exist_ok=True)
print(f'✅ Space repo ready: {SPACE_REPO}')

os.makedirs('/content/hf-space', exist_ok=True)

dockerfile = '''FROM python:3.11-slim
WORKDIR /app
RUN apt-get update && apt-get install -y git curl build-essential && rm -rf /var/lib/apt/lists/*
RUN pip install leann huggingface_hub fastapi uvicorn -q
COPY entrypoint.sh .
COPY health.py .
RUN chmod +x entrypoint.sh
EXPOSE 7860
CMD ["./entrypoint.sh"]
'''

entrypoint = f'''#!/bin/bash
set -e
echo "Downloading LEANN index from HF Dataset..."
python -c "
from huggingface_hub import snapshot_download
import os
snapshot_download(repo_id='{DATASET_REPO}', repo_type='dataset', local_dir='/app/leann-index', token=os.environ.get('HF_TOKEN'))
print('Index downloaded')
"
echo "Starting LEANN MCP server..."
leann serve {INDEX_NAME} --index-dir /app/leann-index/ --port 7860
'''

health = '''from fastapi import FastAPI
import uvicorn
app = FastAPI()
@app.get("/")
def root(): return {"status": "alive", "service": "LEANN Brain MCP"}
@app.get("/health")
def health(): return {"status": "ok"}
if __name__ == "__main__": uvicorn.run(app, host="0.0.0.0", port=7861)
'''

with open('/content/hf-space/Dockerfile', 'w') as f: f.write(dockerfile)
with open('/content/hf-space/entrypoint.sh', 'w') as f: f.write(entrypoint)
with open('/content/hf-space/health.py', 'w') as f: f.write(health)
print('✅ Space files created')

✅ Space repo ready: rajavanchai/glenn
✅ Space files created


In [ ]:
# PHASE 5b — Deploy Space
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
SPACE_REPO = f'{HF_USERNAME}/{SPACE_NAME}'

print('Uploading Space files...')
api.upload_folder(
    folder_path='/content/hf-space',
    repo_id=SPACE_REPO,
    repo_type='space',
    token=HF_TOKEN,
    commit_message='Deploy LEANN MCP server'
)

try:
    api.add_space_secret(SPACE_REPO, 'HF_TOKEN', HF_TOKEN)
    print('✅ HF_TOKEN secret added')
except Exception as e:
    print(f'Add HF_TOKEN manually in Space Settings → Secrets: {e}')

print(f'''
✅ Space deployed!
   URL: https://huggingface.co/spaces/{SPACE_REPO}
   MCP: https://{HF_USERNAME}-{SPACE_NAME}.hf.space
⏳ Wait 3-5 min for Space to build.
''')

No files have been modified since last commit. Skipping to prevent empty commit.


Uploading Space files...
✅ HF_TOKEN secret added

✅ Space deployed!
   URL: https://huggingface.co/spaces/rajavanchai/glenn
   MCP: https://rajavanchai-glenn.hf.space
⏳ Wait 3-5 min for Space to build.



In [ ]:
# PHASE 6 — Claude Terminal Connection Instructions
MCP_URL = f'https://{HF_USERNAME}-{SPACE_NAME}.hf.space'

print('=' * 55)
print('RUN THIS IN YOUR LOCAL TERMINAL:')
print('=' * 55)
print(f'\n  claude mcp add --scope user {INDEX_NAME} --url {MCP_URL}\n')
print('OR add to ~/.claude/mcp_config.json:')
print(f'''{{
  "mcpServers": {{
    "{INDEX_NAME}": {{
      "url": "{MCP_URL}",
      "type": "sse"
    }}
  }}
}}''')
print('\nTEST:')
print(f'  claude "Search my brain for [topic]"')

RUN THIS IN YOUR LOCAL TERMINAL:

  claude mcp add --scope user monarch-personal-brain --url https://rajavanchai-glenn.hf.space

OR add to ~/.claude/mcp_config.json:
{
  "mcpServers": {
    "monarch-personal-brain": {
      "url": "https://rajavanchai-glenn.hf.space",
      "type": "sse"
    }
  }
}

TEST:
  claude "Search my brain for [topic]"
